# 2025년 잔존 미연결 OpenID 수동 판정 검증

EDSS identity 목록에서 2025년까지 관측되는 `unmatched` OpenID와 수동 판정표를 비교한다. 원자료의 0은 결측·비공개·해당 없음과 구분하여 그대로 유지한다.

In [1]:
import csv
from collections import Counter
from pathlib import Path

root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
identity_path = root / 'data/processed/edss_0101_kedi_openid_identity_2009_2025.csv'
review_path = root / 'data/metadata/edss_2025_unmatched_openid_manual_review.csv'

with identity_path.open(encoding='utf-8-sig', newline='') as handle:
    identity = list(csv.DictReader(handle))
with review_path.open(encoding='utf-8-sig', newline='') as handle:
    review = list(csv.DictReader(handle))

target_ids = {row['openid'] for row in identity if row['last_edss_year'] == '2025' and row['identity_status'] == 'unmatched'}
review_ids = [row['open_id'] for row in review]
validation = {
    'target_count': len(target_ids),
    'review_count': len(review),
    'unique_review_ids': len(set(review_ids)),
    'missing_ids': sorted(target_ids - set(review_ids)),
    'extra_ids': sorted(set(review_ids) - target_ids),
    'duplicate_ids': sorted({value for value in review_ids if review_ids.count(value) > 1}),
    'review_order_is_contiguous': [int(row['review_order']) for row in review] == list(range(1, len(review) + 1)),
}
validation

{'target_count': 29,
 'review_count': 29,
 'unique_review_ids': 29,
 'missing_ids': [],
 'extra_ids': [],
 'duplicate_ids': [],
 'review_order_is_contiguous': True}

In [2]:
classification_counts = Counter(row['manual_classification'] for row in review)
action_counts = Counter(
    'include_active' if 'include_in_active' in row['safe_join_action'] else 'exclude_active_performance'
    for row in review
)
{'classification_counts': dict(sorted(classification_counts.items())), 'action_counts': dict(action_counts)}

{'classification_counts': {'closed_school_and_closed_department_residual_id': 6,
  'confirmed_identity_active_id': 4,
  'confirmed_identity_active_name_change_successor_id': 5,
  'confirmed_identity_active_new_id': 4,
  'confirmed_identity_closed_school_residual_id': 1,
  'confirmed_identity_name_change_closed_department_residual_id': 1,
  'confirmed_identity_new_school_zero_activity_id': 3,
  'confirmed_identity_new_school_zero_activity_preopening_id': 2,
  'confirmed_identity_zero_activity_closure_transition_id': 1,
  'confirmed_identity_zero_activity_id': 2},
 'action_counts': {'exclude_active_performance': 16, 'include_active': 13}}